# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

# 0. Document Instructions

## 0.1. Git Version Control

After making modifications, please use Colab's function of "Save in Github to keep changes" (Github authorization required) to commit the changes to the repository (Your own branch if needed).

**NOTE**: When A modifies this notebook, but have not commited to Github, if B opens this notebook, they will see the Github old version.

## 0.2. Handle persistent data storage

Mount Google Drive instead of relying on Colab's temporary run-time storage.

**IMPORTANT:** Although Google Colab allows collaboration of the notebook, data and folders are originally in Junxuan's drive, this could cause issues when refering to the mounted directory path. To allow the code to work for all of us, (who's not junxuan) please create a **short-cut** of Junxuan's shared folder in our own Google Drive, path `MyDrive/`.

In [78]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

: 

# 1. Data Exploration & Preparation

## 1.1. Load and Inspect

### a. Load Data

In [79]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


data_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data/raw")
persona_path = data_dir / "personas.json"
recommendation_path = data_dir / "recommendations.json"
recipe_path = data_dir / "recipes.parquet"

In [80]:
# open files and load data

with open(persona_path, "r", encoding="utf-8") as f:
    personas = json.load(f)

with open(recommendation_path, "r", encoding="utf-8") as f:
    recommendations = json.load(f)

recipes = pd.read_parquet(recipe_path)

### b. Inspect Raw data

In [81]:
%pip install -q spacy pyarrow nltk
!python -m spacy download en_core_web_sm

import sys

helper_search_paths = [Path.cwd(), Path.cwd() / "notebooks"]
helper_dir = next(
    (
        path
        for path in helper_search_paths
        if (path / "data_exploration_helpers.py").exists()
    ),
    None,
)

if helper_dir is None:
    raise FileNotFoundError(
        "Could not find data_exploration_helpers.py from the current working directory."
    )

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from data_exploration_helpers import (
    add_text_items,
    as_lemma_list,
    build_matching_resources,
    color_for_combination,
    compute_coverage,
    derive_nutrition_terms,
    focus_combination,
    is_empty_value,
    make_lemma_table,
    make_token_table,
    remove_leading_title,
)


In [82]:
print("personas")
print("type:", type(personas))
print("length:", len(personas))

print(json.dumps(personas[0],indent=2))

In [83]:
print("\nrecommendations")
print("type:", type(recommendations))
print("length:", len(recommendations))

print(json.dumps(recommendations[0],indent=2))

In [84]:
print("\nrecipes")
print("type:", type(recipes))
print("shape:", recipes.shape)

display(recipes.head())


### c. Organize Raw Data as DataFrames and Further Inspect

In [85]:
# pd display setting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

#### c.1. persona

In [86]:
personas_df = pd.json_normalize(personas)

print(personas_df.shape)
display(personas_df.head())

In [87]:
personas_df.info()

In [88]:
display(pd.DataFrame({
    "missing": personas_df.isna().sum(),
    "empty_list_or_text": personas_df.apply(lambda col: col.map(is_empty_value).sum())
}))

In [89]:
print("unique personas:", personas_df["id"].nunique())
print("duplicate ids:", personas_df["id"].duplicated().sum())

In [90]:
# persona distribution

distribution_columns = {
    "Gender": "profile.gender",
    "Dietary Goal": "profile.dietary_goal",
    "Activity Level": "profile.activityLevel",
    "Dietary Restriction": "profile.dietaryProfile.dietaryRestrictions.selected",
    "Food Allergy": "profile.dietaryProfile.foodAllergies.selected",
    "Health Condition": "profile.dietaryProfile.healthConditions.selected"
}

persona_distribution = {}

for name, column in distribution_columns.items():
    values = personas_df[column]

    if values.apply(lambda x: isinstance(x, list)).all():
        values = values.explode()

    counts = values.value_counts()
    persona_distribution[name] = pd.Series(
        [f"{category}: {count}" for category, count in counts.items()]
    )

display(pd.DataFrame(persona_distribution).fillna(""))

#### c.2. recommendation

In [91]:
# rank is the position in the original recommendation list (store rank for future tasks).
# persona -> recommendations

recommendation_rows = []

for item in recommendations:
    persona_id = item["persona"]["id"]

    for rank, rec in enumerate(item["recommendations"], start=1):
        recommendation_rows.append({
            **rec,
            "persona_id": persona_id,
            "rank": rank
        })

recommendations_df = pd.DataFrame(recommendation_rows)

print(recommendations_df.shape)
display(recommendations_df.head())

In [49]:
recommendations_df.info()

In [92]:
print(recommendations_df.columns.tolist())

In [94]:
# show no selection separately from missing data. dietary type
restriction_values = personas_df["profile.dietaryProfile.dietaryRestrictions.selected"]
display(restriction_values.map(
    lambda x: ["No selection"] if isinstance(x, list) and not x else x
).explode().fillna("Missing").value_counts())

In [95]:
# recommentation number for personas
# include personas with zero recommendations also
rec_counts = recommendations_df.groupby("persona_id").size().reindex(personas_df["id"], fill_value=0)
display(rec_counts.describe())

In [96]:
display(rec_counts.sort_values())

In [97]:
print("total recommendations:", len(recommendations_df))
print("personas:", recommendations_df["persona_id"].nunique())
print("unique recipes:", recommendations_df["recipe_id"].nunique())

In [98]:
pair_columns = ["persona_id", "recipe_id"]
duplicate_recs = recommendations_df.duplicated(subset=pair_columns, keep=False)

print("rows belonging to repeated persona-recipe pairs:", duplicate_recs.sum())
print("repeated persona-recipe pairs:", recommendations_df.loc[duplicate_recs, pair_columns].drop_duplicates().shape[0])
print("extra rows beyond one per persona-recipe pair:", recommendations_df.duplicated(subset=pair_columns).sum())

In [99]:
display(
    recommendations_df.loc[
        duplicate_recs,
        ["persona_id", "rank", "recipe_id", "title", "explanation"]
    ].sort_values(["persona_id", "recipe_id"])
)

In [100]:
print(
    "unique persona-recipe pairs:",
    recommendations_df[
        ["persona_id", "recipe_id"]
    ].drop_duplicates().shape[0]
)

#### c.3. recipe

In [101]:
print(recommendations_df["recipe_id"].dtype)
print(recipes["recipe_id"].dtype)

In [102]:
# join 2 df

recommendations_df["recipe_id"] = recommendations_df["recipe_id"].astype(str)
recipes["recipe_id"] = recipes["recipe_id"].astype(str)

matched = recommendations_df["recipe_id"].isin(recipes["recipe_id"])

print("matched:", matched.sum())
print("unmatched:", (~matched).sum())
print("match rate:", matched.mean())

In [103]:
nutrition_df = pd.json_normalize(recommendations_df["nutrition"])
nutrition_df.index = recommendations_df.index

# recommendation protein/carbs/fat are stored as strings such as "13.3g".

for column in ["calories", "protein", "carbs", "fat"]:
    values = nutrition_df[column].astype("string").str.strip()
    if column != "calories":
        values = values.str.replace(r"\s*g$", "", regex=True)
    nutrition_df[column] = pd.to_numeric(values, errors="coerce")

print("missing or unparseable recommendation nutrients:")
display(nutrition_df.isna().sum())
display(nutrition_df.head())

In [61]:
recommendations_prepared_df = pd.concat(
    [recommendations_df.drop(columns="nutrition"), nutrition_df.add_prefix("rec_")],
    axis=1
)

In [62]:
# many recommendations may reference one source recipe.

merged_df = recommendations_prepared_df.merge(
    recipes,
    on="recipe_id",
    how="left",
    suffixes=("_rec", "_recipe"),
    indicator=True,
    validate="many_to_one"
)

print(merged_df.shape)
display(merged_df.head())

In [63]:
# check if all matched successfully

print(
    "recommendations without recipe data:",
    merged_df["_merge"].eq("left_only").sum()
)

## 1.2. Extract all explanations and compute basic statistics

### a. Extract all explanations

In [64]:
import re

explanations_df = recommendations_df[
    ["persona_id", "recipe_id", "rank", "title", "explanation"]
].copy()

display(explanations_df.head())


### b. Compute Length (words/characters)

In [65]:
# add length

word_pattern = re.compile(r"\b\w+(?:[-'’]\w+)*\b") # to filter non-standard texts

explanations_df["word_count"] = explanations_df["explanation"].str.findall(word_pattern).str.len()
explanations_df["character_count"] = explanations_df["explanation"].str.len()

display(explanations_df.head())

In [66]:
# distribution

analysis_df = explanations_df.copy()

length_summary = analysis_df[["word_count", "character_count"]].describe().T.rename(
    columns={"50%": "median"}
)
display(length_summary.round(2))

In [67]:
# plot

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, column, label in zip(axes, ["word_count", "character_count"], ["Words", "Characters"]):
    ax.hist(analysis_df[column], bins=20, edgecolor="white")
    ax.set(xlabel=label, ylabel="Recommendation entries", title=f"Explanation length ({label.lower()})")
plt.tight_layout()
plt.show()

### b-c-1 Preprocessing raw data for further analysis

 (Run ONLY Once, and store the result for future use)

compose records for recommendation explanations, recipes and personas (raw text for now).

In [69]:
# make records for explanations, recipes, and personas ############

# One explanation row per recommendation.
explanation_records = []

for item in recommendations:
    persona_id = str(item["persona"]["id"])

    for rank, recommendation in enumerate(
        item["recommendations"],
        start=1
    ):
        recipe_id = str(recommendation["recipe_id"])
        recommendation_id = f"{persona_id}_rank_{rank:02d}"

        add_text_items(
            explanation_records,
            entity_id=recommendation_id,
            source_field="explanation", # register
            value=recommendation["explanation"], # explanation text
            lemma_column="explanation_lemmas", # specify column name where stores lemmas
            keep_numbers=True,
            persona_id=persona_id, # who's recommendation is this
            recipe_id=recipe_id, # recommended recipe's id
            rank=rank, # the recommendation rank
            recommendation_id=recommendation_id
        )

# Recipe titles, ingredients, and tags.
recipe_records = []

# generate records for parts of recipes separately
for _, row in recipes.drop_duplicates("recipe_id").iterrows():
    recipe_id = str(row["recipe_id"])

    # recipe title as record
    add_text_items(
        recipe_records,
        entity_id=recipe_id,
        source_field="recipe_title",
        value=row["title"],
        lemma_column="recipe_text_lemmas", # specify column name where stores lemmas
        keep_numbers=True,
        recipe_id=recipe_id
    )

    # recipe ingredient as record
    add_text_items(
        recipe_records,
        entity_id=recipe_id,
        source_field="recipe_ingredient",
        value=row["ingredients_title"],
        lemma_column="recipe_text_lemmas", # specify column name where stores lemmas
        keep_numbers=False,
        recipe_id=recipe_id,
        remove_ingredient_quantities=True
    )

    # recipe tags as record
    add_text_items(
        recipe_records,
        entity_id=recipe_id,
        source_field="recipe_tag",
        value=row["tags"],
        lemma_column="recipe_text_lemmas", # specify column name where stores lemmas
        keep_numbers=True,
        recipe_id=recipe_id
    )



# Persona descriptions, goals, preferences, restrictions, allergies,
# and health conditions.
persona_records = []

persona_fields = {
    "description": "persona_description",
    "profile.dietary_goal": "dietary_goal",
    "profile.dietaryProfile.dietaryRestrictions.selected":
        "dietary_restriction",
    "profile.dietaryProfile.dietaryRestrictions.other":
        "dietary_restriction_other",
    "profile.dietaryProfile.foodAllergies.selected":
        "food_allergy",
    "profile.dietaryProfile.foodAllergies.other":
        "food_allergy_other",
    "profile.dietaryProfile.healthConditions.selected":
        "health_condition",
    "profile.dietaryProfile.healthConditions.other":
        "health_condition_other",
    "profile.likedIngredients": "liked_ingredient",
    "profile.dislikedIngredients": "disliked_ingredient",
    "profile.favoriteCuisines": "favorite_cuisine"
}

for _, row in personas_df.iterrows():
    persona_id = str(row["id"])

    for column, source_field in persona_fields.items():
        add_text_items(
            persona_records,
            entity_id=persona_id,
            source_field=source_field,
            value=row[column],
            lemma_column="persona_text_lemmas",
            keep_numbers=True,
            persona_id=persona_id
        )


compose lemmatized text of the former records

In [70]:
explanation_lemmas_df = make_lemma_table(explanation_records)
recipe_lemmas_df = make_lemma_table(recipe_records)
persona_lemmas_df = make_lemma_table(persona_records)

explanation_lemmas_df["rank"] = (
    explanation_lemmas_df["rank"].astype("Int64")
)

assert explanation_lemmas_df["recommendation_id"].is_unique
assert len(explanation_lemmas_df) == len(recommendations_df)

save results for future use

In [71]:
# store the processed text for future use
processed_dir = Path(
    "/content/drive/MyDrive/XFoodRec_Project_Data/processed"
)
processed_dir.mkdir(parents=True, exist_ok=True)


explanation_lemmas_df.to_parquet(
    processed_dir / "explanation_lemmas.parquet",
    index=False
)
recipe_lemmas_df.to_parquet(
    processed_dir / "recipe_lemmas.parquet",
    index=False
)
persona_lemmas_df.to_parquet(
    processed_dir / "persona_lemmas.parquet",
    index=False
)

print("Saved processed files to:", processed_dir)
print("Explanation records:", len(explanation_lemmas_df))
print("Recipe text items:", len(recipe_lemmas_df))
print("Persona text items:", len(persona_lemmas_df))

display(
    explanation_lemmas_df[
        [
            "recommendation_id",
            "persona_id",
            "recipe_id",
            "rank",
            "explanation_lemmas"
        ]
    ].head()
)

display(
    recipe_lemmas_df.loc[
        recipe_lemmas_df["source_field"].eq("recipe_ingredient"),
        ["recipe_id", "recipe_text_lemmas"]
    ].head()
)

Save token tables and nutrition field names

In [72]:
# build the three token tables from the same records as the lemma tables.
recommendation_tokens_df = make_token_table(explanation_records)
recipe_tokens_df = make_token_table(recipe_records)
persona_tokens_df = make_token_table(persona_records)

for filename, table in [
    ("recommendation_tokens.parquet", recommendation_tokens_df),
    ("recipe_tokens.parquet", recipe_tokens_df),
    ("persona_tokens.parquet", persona_tokens_df),
]:
    table.to_parquet(processed_dir / filename, index=False)
    print(f"Saved {filename}: {len(table)} text items")

display(recommendation_tokens_df[["recommendation_id", "text", "tokens"]].head())

In [73]:
# collect actual nutrition keys rather than maintaining a separate word list.
import json

nutrition_fields = list(dict.fromkeys(
    field
    for item in recommendations
    for recommendation in item["recommendations"]
    for field in recommendation["nutrition"]
))
nutrition_fields_path = processed_dir / "nutrition_fields.json"
with nutrition_fields_path.open("w", encoding="utf-8") as f:
    json.dump(nutrition_fields, f, ensure_ascii=False, indent=2)

print("Saved nutrition fields:", nutrition_fields)

### b-c-2 reload preprocessed data

In [74]:
processed_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data/processed")

recommendation_tokens_df = pd.read_parquet(processed_dir / "recommendation_tokens.parquet")
recipe_tokens_df = pd.read_parquet(processed_dir / "recipe_tokens.parquet")
persona_tokens_df = pd.read_parquet(processed_dir / "persona_tokens.parquet")
explanation_lemmas_df = pd.read_parquet(processed_dir / "explanation_lemmas.parquet")
recipe_lemmas_df = pd.read_parquet(processed_dir / "recipe_lemmas.parquet")
persona_lemmas_df = pd.read_parquet(processed_dir / "persona_lemmas.parquet")
nutrition_fields_path = processed_dir / "nutrition_fields.json"

print("Explanations:", len(recommendation_tokens_df))
print("Recipe text items:", len(recipe_tokens_df))
print("Persona text items:", len(persona_tokens_df))

Persisted file schemas and shared-helper behavior are documented in [`data_exploration_reference.md`](data_exploration_reference.md).

In [75]:
recipe_table = recipes if "recipes" in globals() else pd.read_parquet(recipe_path)
with open(nutrition_fields_path, "r", encoding="utf-8") as f:
    nutrition_fields = json.load(f)
matching_resources = build_matching_resources(persona_lemmas_df, recipe_lemmas_df, recipe_table, nutrition_fields)
print("Nutrients recorded in the data:", matching_resources["nutrient_terms"])
print("Health label values (goals, diets, conditions):", len(matching_resources["health_label_values"]))
print("Personas with values:", len(matching_resources["persona_values"]))
print("Recipes with ingredient names:", len(matching_resources["ingredient_names"]),
      "| food words:", len(matching_resources["food_terms"]))

### c. Distribution of explanation types (ingredient-, nutrition-, taste-focused)

In [76]:
from matplotlib.patches import Patch

# nutrition field names
if "nutrition_df" in globals():
    nutrition_columns = list(nutrition_df.columns)

    with open(nutrition_fields_path, "w", encoding="utf-8") as f:
        json.dump(nutrition_columns, f, ensure_ascii=False, indent=2)

elif nutrition_fields_path.exists():
    with open(nutrition_fields_path, "r", encoding="utf-8") as f:
        nutrition_columns = json.load(f)

elif "recipes" in globals():
    nutrition_columns = [
        column
        for column in recipes.columns
        if "_per_serving" in str(column).lower()
        or "_per_100g" in str(column).lower()
    ]

else:
    raise FileNotFoundError(
        "Nutrition field names are unavailable. "
        "Run the data-loading cells once so they can be saved."
    )

nutrition_terms = derive_nutrition_terms(nutrition_columns)

taste_terms = {
    "taste", "flavor", "flavour", "delicious", "tasty",
    "savory", "savoury", "sweet", "spicy", "tangy",
    "creamy", "crunchy", "crispy", "tender", "juicy",
    "aromatic", "refreshing", "hearty", "satisfy",
    "comfort", "warm", "mild", "bitter", "sour", "smoky",
    "texture"
}

# occur in ingredient descriptions, but are not ingredients.
ingredient_noise_terms = {
    "taste", "your", "favorite", "favourite", "optional",
    "needed", "desired", "garnish", "serve", "serving",
    "adjust", "approximately"
}

ingredients_by_recipe = {}
titles_by_recipe = {}

for row in recipe_lemmas_df.to_dict("records"):
    recipe_id = str(row["recipe_id"])
    source_field = row["source_field"]
    lemmas = as_lemma_list(row["recipe_text_lemmas"])

    if source_field == "recipe_ingredient":
        clean_ingredients = {
            lemma
            for lemma in lemmas
            if lemma not in ingredient_noise_terms
        }
        ingredients_by_recipe.setdefault(recipe_id, set()).update(
            clean_ingredients
        )

    elif source_field == "recipe_title":
        titles_by_recipe[recipe_id] = lemmas

# match each explanation to focus vocabularies
analysis_df = explanation_lemmas_df.copy()

ingredient_matches = []
nutrition_matches = []
taste_matches = []

for row in analysis_df.to_dict("records"):
    recipe_id = str(row["recipe_id"])
    explanation = as_lemma_list(row["explanation_lemmas"])

    explanation_body = remove_leading_title(
        explanation,
        titles_by_recipe.get(recipe_id, [])
    )

    recipe_ingredients = ingredients_by_recipe.get(recipe_id, set())
    explanation_terms = set(explanation_body)

    ingredient_matches.append(
        sorted(explanation_terms & recipe_ingredients)
    )
    nutrition_matches.append(
        sorted(explanation_terms & nutrition_terms)
    )
    taste_matches.append(
        sorted(explanation_terms & taste_terms)
    )

analysis_df["ingredient_matches"] = ingredient_matches
analysis_df["nutrition_matches"] = nutrition_matches
analysis_df["taste_matches"] = taste_matches

analysis_df["ingredient_focused"] = analysis_df["ingredient_matches"].map(bool)
analysis_df["nutrition_focused"] = analysis_df["nutrition_matches"].map(bool)
analysis_df["taste_focused"] = analysis_df["taste_matches"].map(bool)

focus_columns = [
    "ingredient_focused",
    "nutrition_focused",
    "taste_focused"
]
focus_labels = {
    "ingredient_focused": "Ingredient",
    "nutrition_focused": "Nutrition",
    "taste_focused": "Taste"
}

total_explanations = len(analysis_df)

if total_explanations == 0:
    raise ValueError("No explanation records were loaded.")

if not analysis_df[focus_columns].any(axis=None):
    raise ValueError(
        "No focus matches were found. Check the loaded lemmas and vocabularies."
    )

# summary by focus
type_summary = pd.DataFrame({
    "count": analysis_df[focus_columns].sum().astype(int),
    "percent": (
        analysis_df[focus_columns].mean().mul(100).round(1)
    )
})
type_summary.index = [
    focus_labels[column]
    for column in type_summary.index
]

display(type_summary)

analysis_df["focus_combination"] = analysis_df.apply(
    focus_combination,
    axis=1
)

combination_summary = (
    analysis_df["focus_combination"]
    .value_counts()
    .rename_axis("focus_combination")
    .to_frame("count")
)
combination_summary["percent"] = (
    combination_summary["count"]
    .div(total_explanations)
    .mul(100)
    .round(1)
)

display(combination_summary)

focus_colors = {
    "Ingredient": "#E76F51",
    "Nutrition": "#2A9D8F",
    "Taste": "#E9C46A"
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    gridspec_kw={"width_ratios": [0.8, 1.5]}
)
# Total count and percentage in each focus category.
left_labels = ["Ingredient", "Nutrition", "Taste"]

left_counts = [
    int(analysis_df[column].sum())
    for column in focus_columns
]

left_percentages = [
    100 * count / total_explanations
    for count in left_counts
]

bars = axes[0].bar(
    left_labels,
    left_counts,
    color=[focus_colors[label] for label in left_labels],
)

axes[0].bar_label(
    bars,
    labels=[
        f"{count} ({percentage:.1f}%)"
        for count, percentage in zip(
            left_counts,
            left_percentages,
        )
    ],
    padding=3,
)

axes[0].set_title("Explanations by focus")
axes[0].set_ylabel("Number of explanations")
axes[0].set_ylim(0, max(left_counts) * 1.18)
axes[0].tick_params(axis="x", rotation=15)

# Show counts and percentages for exact combinations.
plot_data = combination_summary.sort_values("count")
combination_labels = plot_data.index.tolist()
combination_counts = plot_data["count"].tolist()
combination_percentages = plot_data["percent"].tolist()

bars = axes[1].barh(
    combination_labels,
    combination_counts,
    color=[
        color_for_combination(label)
        for label in combination_labels
    ],
)

axes[1].bar_label(
    bars,
    labels=[
        f"{count} ({percentage:.1f}%)"
        for count, percentage in zip(
            combination_counts,
            combination_percentages,
        )
    ],
    padding=3,
)

axes[1].set_title("Exact focus combinations")
axes[1].set_xlabel("Number of explanations")
axes[1].set_xlim(0, max(combination_counts) * 1.35)

axes[1].legend(
    handles=[
        Patch(facecolor=color, label=label)
        for label, color in focus_colors.items()
    ],
    title="Focus colors",
    loc="lower right"
)

fig.suptitle(
    f"Explanation focus distribution (n={total_explanations})"
)
plt.tight_layout()
plt.show()

# Inspect matches to confirm that ingredient noise terms are gone.
display(
    analysis_df[
        [
            "recommendation_id",
            "ingredient_matches",
            "nutrition_matches",
            "taste_matches",
            "focus_combination"
        ]
    ].head(15)
)

### d. Coverage of user preferences vs. health justification

Each explanation is checked for two things: whether it refers to the user's preferences, and whether it gives a health justification. One explanation can count as both, one, or neither. Only things that exist in the data count.

| What the explanation mentions | Counted as |
|---|---|
| The user's own liked or disliked ingredients or favourite cuisines (e.g. "lentils, which you enjoy", or "features chicken" for a user who likes chicken) | Preference |
| A nutrient recorded in the recipe data, with or without numbers (e.g. "high in protein", "33.1 g of protein", "low in sodium") | Health |
| A goal, diet or health condition from the user profiles (e.g. "weight loss", "vegan", "low sodium") | Health |
| An allergen, only for the user who has that allergy | Health |
| General wording that names none of the above (e.g. "fits your preferences", "suits your dietary needs", "healthy") | Neither (left to the transparency analysis) |

The recipe title is removed from the explanation before matching, so words that only appear in the title are not counted.

In [77]:
coverage_df = compute_coverage(
    explanation_lemmas_df,
    matching_resources,
)

# Add coverage results to the main analysis table.
if "analysis_df" not in globals():
    analysis_df = explanation_lemmas_df.copy()

existing_coverage_columns = [
    column
    for column in coverage_df.columns
    if (
        column != "recommendation_id"
        and column in analysis_df.columns
    )
]

analysis_df = (
    analysis_df
    .drop(columns=existing_coverage_columns)
    .merge(
        coverage_df,
        on="recommendation_id",
        how="left",
        validate="one_to_one",
    )
)

# Summarize coverage groups.
total_explanations = len(coverage_df)

coverage_order = [
    "Both",
    "Preference only",
    "Health only",
    "Neither",
]

coverage_summary = (
    coverage_df["coverage_group"]
    .value_counts()
    .reindex(coverage_order, fill_value=0)
    .rename("count")
    .to_frame()
)

coverage_summary["percent"] = (
    100
    * coverage_summary["count"]
    / total_explanations
).round(1)

display(coverage_summary)

# Summarize the evidence used for classification.
has_nutrition_evidence = (
    coverage_df["nutrition_evidence"].map(bool)
)
has_health_evidence = (
    coverage_df["health_evidence"].map(bool)
)

evidence_summary = pd.DataFrame(
    {
        "explanations": [
            coverage_df["preference_covered"].sum(),
            coverage_df["health_justification"].sum(),
            has_nutrition_evidence.sum(),
            has_health_evidence.sum(),
        ]
    },
    index=[
        "Preference: user's own ingredient or cuisine",
        "Health justification (any)",
        "  nutrient recorded in the recipe data",
        "  goal, diet, allergy or condition",
    ],
)

evidence_summary["percent"] = (
    100
    * evidence_summary["explanations"]
    / total_explanations
).round(1)

display(evidence_summary)

# Plot the coverage groups.
coverage_colors = {
    "Both": "#2A9D8F",
    "Preference only": "#E9C46A",
    "Health only": "#E76F51",
    "Neither": "#B8B8B8",
}

plot_order = coverage_order[::-1]
plot_counts = coverage_summary.loc[plot_order, "count"]
plot_colors = [
    coverage_colors[group]
    for group in plot_order
]

fig, ax = plt.subplots(figsize=(7, 3.5))

bars = ax.barh(
    plot_order,
    plot_counts,
    color=plot_colors,
)

ax.bar_label(bars, padding=3)
ax.set_xlabel("Number of explanations")
ax.set_title(
    "Coverage of user preferences vs. health justification "
    f"(n={total_explanations})"
)
ax.set_xlim(
    0,
    coverage_summary["count"].max() * 1.15,
)

plt.tight_layout()
plt.show()